# OLMo-2 1B DPO Activation Oracle Demo

This notebook uses the trained activation oracle to ask natural-language questions
about the internal activations of `allenai/OLMo-2-0425-1B-DPO`.

**Flow:**
1. Load the subject model (whose activations we want to inspect)
2. Run a prompt through it and collect residual-stream activations at a chosen layer
3. Pass those activations to the oracle (same base model + oracle LoRA) via a hook at layer 1
4. Ask the oracle an arbitrary natural-language question

## 1. Imports

In [1]:
import os
os.chdir("/workspace/activation_oracles")
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
from peft import PeftModel

from nl_probes.utils.common import load_model, load_tokenizer, layer_percent_to_layer
from nl_probes.utils.activation_utils import collect_activations_multiple_layers, get_hf_submodule
from nl_probes.utils.dataset_utils import create_training_datapoint
from nl_probes.utils.eval import run_evaluation

/workspace/activation_oracles/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

The oracle was trained on layers at **25%, 50%, 75%** of model depth.
Pick one of those for `LAYER_PERCENT`. The layer number is embedded in the oracle prompt prefix,
so it must match a layer the oracle was trained on.

In [2]:
SUBJECT_MODEL_NAME = "allenai/OLMo-2-0425-1B-DPO"
SUBJECT_MODEL_REV  = None
TOKENIZER_NAME     = "allenai/OLMo-2-0425-1B-DPO"

ORACLE_LORA_PATH   = "downloaded_adapter/olmo2_1b_dpo_checkpoint_oracle_v1"

LAYER_PERCENT      = 50    # which layer to inspect — must be one of [25, 50, 75]
INJECTION_LAYER    = 1     # always 1 — where the oracle hook injects activations
NUM_POSITIONS      = 10    # number of token positions to pass to the oracle

DTYPE  = torch.bfloat16
DEVICE = torch.device("cuda")

## 3. Load the model

The subject model and oracle share the same base weights. We load once and switch between
base (for activation collection) and oracle LoRA (for querying) via `disable_adapters` / `set_adapter`.

In [3]:
tokenizer  = load_tokenizer(TOKENIZER_NAME)
base_model = load_model(SUBJECT_MODEL_NAME, DTYPE, model_revision=SUBJECT_MODEL_REV)

# Load oracle LoRA — this creates a PeftModel with the oracle adapter as "default"
model = PeftModel.from_pretrained(base_model, ORACLE_LORA_PATH, is_trainable=False)

act_layer = layer_percent_to_layer(SUBJECT_MODEL_NAME, LAYER_PERCENT, SUBJECT_MODEL_REV)
print(f"Inspecting layer {act_layer} ({LAYER_PERCENT}% of model depth)")

📦 Loading tokenizer...


🧠 Loading model...


Inspecting layer 8 (50% of model depth)


## 4. Define the subject context

This is the text the subject model processes. The oracle will answer questions about
what's happening in the model's residual stream at the chosen layer while processing this text.

In [4]:
CONTEXT_MESSAGES = [
    {"role": "user", "content": "The philosopher who drank hemlock taught a student who founded an academy. That student's most famous pupil was"},
]

context_text = tokenizer.apply_chat_template(
    CONTEXT_MESSAGES,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("Context fed to subject model:")
print(repr(context_text))

Context fed to subject model:
"<|endoftext|><|user|>\nThe philosopher who drank hemlock taught a student who founded an academy. That student's most famous pupil was\n<|assistant|>\n"


## 5. Collect activations from the subject model

We run the context through the base model (adapters disabled) and capture
residual-stream activations at `act_layer`.

In [5]:
inputs_BL = tokenizer(
    [context_text],
    return_tensors="pt",
    add_special_tokens=False,
).to(DEVICE)

context_ids = inputs_BL["input_ids"][0].tolist()
print(f"Context length: {len(context_ids)} tokens")

# use_lora=True traverses PeftModel correctly: model.base_model.model.model.layers[layer]
submodules = {act_layer: get_hf_submodule(model, act_layer, use_lora=True)}

# Collect activations from the base model (oracle adapter disabled)
with model.disable_adapter():
    with torch.no_grad():
        acts_by_layer = collect_activations_multiple_layers(
            model, submodules, inputs_BL, min_offset=None, max_offset=None
        )

acts_LD = acts_by_layer[act_layer][0]   # [L, D]
print(f"Activation shape: {acts_LD.shape}")

Context length: 33 tokens


Activation shape: torch.Size([33, 2048])


## 6. Ask the oracle a question

Choose which token positions to pass and what question to ask.
We take the last `NUM_POSITIONS` tokens by default (most informative for generation context).

In [6]:
QUESTION = "Can you name which person the model is thinking about?"

# Which positions to pass — last NUM_POSITIONS tokens
n_pos     = min(NUM_POSITIONS, acts_LD.shape[0])
positions = list(range(acts_LD.shape[0] - n_pos, acts_LD.shape[0]))
acts_KD   = acts_LD[positions]   # [K, D]

print(f"Passing positions {positions}")
print(f"Tokens at those positions: {[tokenizer.decode([context_ids[p]]) for p in positions]}")

Passing positions [23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
Tokens at those positions: [' most', ' famous', ' pupil', ' was', '\n', '<', '|', 'assistant', '|', '>\n']


In [7]:
datapoint = create_training_datapoint(
    datapoint_type="demo",
    prompt=QUESTION,
    target_response="N/A",
    layer=act_layer,
    num_positions=n_pos,
    tokenizer=tokenizer,
    acts_BD=acts_KD,
    feature_idx=-1,
    context_input_ids=None,
    context_positions=None,
)

injection_submodule = get_hf_submodule(model, INJECTION_LAYER, use_lora=True)

responses = run_evaluation(
    eval_data=[datapoint],
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,   # oracle adapter already loaded as "default"
    eval_batch_size=1,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 100},
)

print("Oracle response:")
print(responses[0].api_response)

Evaluating model:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Oracle response:
The assistant is thinking about the philosopher who was most famous for his pupil.


## 7. Sweep over individual token positions

Query the oracle once per token — each time passing a single token's activation.
This lets you see which positions carry the most interesting signal.

In [8]:
QUESTION_PER_TOKEN = "Can you name who the model is thinking about?"
# QUESTION_PER_TOKEN = "Can you name which person the model is thinking about?"

per_token_datapoints = []
for i in range(len(context_ids)):
    dp = create_training_datapoint(
        datapoint_type="demo",
        prompt=QUESTION_PER_TOKEN,
        target_response="N/A",
        layer=act_layer,
        num_positions=1,
        tokenizer=tokenizer,
        acts_BD=acts_LD[i:i+1],
        feature_idx=i,
        context_input_ids=None,
        context_positions=None,
    )
    per_token_datapoints.append(dp)

per_token_responses = run_evaluation(
    eval_data=per_token_datapoints,
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,   # oracle adapter already loaded as "default"
    eval_batch_size=16,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 30},
)

print(f"{'Pos':>4}  {'Token':<20}  Response")
print("-" * 70)
for i, resp in enumerate(per_token_responses):
    tok = tokenizer.decode([context_ids[i]]).replace("\n", "\\n")
    print(f"{i:>4}  {tok:<20}  {resp.api_response}")

Evaluating model:   0%|          | 0/3 [00:00<?, ?it/s]

Evaluating model:  33%|███▎      | 1/3 [00:00<00:01,  1.69it/s]

Evaluating model:  67%|██████▋   | 2/3 [00:02<00:01,  1.34s/it]

Evaluating model: 100%|██████████| 3/3 [00:02<00:00,  1.19it/s]

Evaluating model: 100%|██████████| 3/3 [00:02<00:00,  1.11it/s]

 Pos  Token                 Response
----------------------------------------------------------------------
   0  <|endoftext|>         The model is thinking about a person who is a professional photographer.
   1  <                     The model is thinking about her long-lost twin.
   2  |                     The model is thinking about her family.
   3  user                  The assistant is thinking about the model who is a professional photographer.
   4  |                     The assistant is thinking about the person who recently moved to a new city.
   5  >\n                   The assistant is thinking about the person who recently moved to a new city.
   6  The                   The assistant is thinking about a person who is always late.
   7   philosopher          The philosopher is thinking about the concept of time.
   8   who                  The model is thinking about the life of the philosopher NAME_1.
   9   drank                The model is thinking about the famous 

## 8. Ask the oracle: what is the goal of the model?

Using the same activations collected from the prompt, ask the oracle a higher-level question
about the model's overall intent.

In [9]:
goal_datapoint = create_training_datapoint(
    datapoint_type="demo",
    prompt="What is the goal of the model?",
    target_response="N/A",
    layer=act_layer,
    num_positions=n_pos,
    tokenizer=tokenizer,
    acts_BD=acts_KD,
    feature_idx=-1,
    context_input_ids=None,
    context_positions=None,
)

goal_responses = run_evaluation(
    eval_data=[goal_datapoint],
    model=model,
    tokenizer=tokenizer,
    submodule=injection_submodule,
    device=DEVICE,
    dtype=DTYPE,
    global_step=-1,
    lora_path=None,
    eval_batch_size=1,
    steering_coefficient=1.0,
    generation_kwargs={"do_sample": False, "max_new_tokens": 100},
)

print("Oracle response:")
print(goal_responses[0].api_response)

Evaluating model:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]

Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]

Oracle response:
The goal of the model is to provide a detailed and accurate answer to the user's question about the most famous pupil of NAME_1.
